# Deep Research Agent with Oracle AI Agent Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openai/openai-cookbook/blob/main/examples/agents_sdk/deep_research_openai_agents.ipynb) [![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

**Framework:** OpenAI Agents SDK · **Web search:** Tavily · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook demonstrates an **Oracle-backed Session pattern for the OpenAI Agents SDK**. It is not a general-purpose OpenAI memory implementation. The Agents SDK `Session` interface stores short-term conversation history for the current agent loop; durable research findings are saved separately through explicit memory tools backed by Oracle AI Agent Memory.

The example builds a **deep research agent** for human genome exploration. The agent uses Tavily for live web research, stores OpenAI Agents SDK session items in Oracle AI Database, and saves durable findings through a dedicated `save_research_finding` tool.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational | Customer support, coding copilot, chat UIs |
| **Deep Research** | Multi-step autonomous investigation | Literature review, market research, technical scoping |
| **Workflow** | Predetermined sequence of steps, often with conditional branches | Approval pipelines, compliance checks, structured triage |

> **📌 This notebook focuses on the Deep Research mode.**
>
> A deep-research agent plans, retrieves, synthesises, and accumulates findings over long-horizon investigations. Memory is central, but not all memory has the same lifecycle: session items support short-term continuity, while durable findings are curated facts the agent can reuse across sessions.

## What You'll Learn

- How to implement the **OpenAI Agents SDK `Session` protocol** against a custom backend (Oracle AI Agent Memory)
- How to wrap **Tavily** as a `function_tool` the agent can call
- How to store long-lived research findings as durable **memories** (separate from short-term conversation history)
- How to verify memory continuity across sessions — the agent can pick up where it left off

> **💡 Key insight:** Short-term memory (conversation history for the current run) and long-term memory (durable findings across runs) are different access patterns over the same store. We use `Session` for the former and `add_memory()` for the latter.

## Prerequisites

- **Python 3.10 or 3.11** recommended for this notebook
- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **`OPENAI_API_KEY`** for the agent's LLM and embeddings
- **`TAVILY_API_KEY`** for web search (free tier available at [tavily.com](https://tavily.com))
- The `oracleagentmemory`, `openai-agents`, `tavily-python`, `python-dotenv`, and `nest_asyncio` packages installed in the active kernel's environment

Create a `.env` file next to the notebook or set these variables in your shell:

```env
OPENAI_API_KEY=
TAVILY_API_KEY=
ORACLE_USER=
ORACLE_PASSWORD=
ORACLE_DSN=
```

The notebook also accepts the legacy aliases `DB_USER`, `DB_PASSWORD`, and `DB_CONNECT_STRING`, but the Oracle-style names above are preferred.

## 1. Install dependencies

We need four packages: `oracleagentmemory` and `litellm` (memory + embeddings), `openai-agents` (agent framework), `tavily-python` (web search), and `nest_asyncio` (so `Runner.run` works cleanly inside a Jupyter event loop).

In [1]:
%pip install -q oracleagentmemory litellm openai-agents tavily-python nest_asyncio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configure API keys and Oracle connection

Load configuration from `.env` or environment variables. This notebook intentionally does **not** provide default credentials. If a required value is missing, it prompts you for it rather than embedding secrets in the notebook.

Required values:

- `OPENAI_API_KEY`
- `TAVILY_API_KEY`
- `ORACLE_USER` or `DB_USER`
- `ORACLE_PASSWORD` or `DB_PASSWORD`
- `ORACLE_DSN` or `DB_CONNECT_STRING`

In [2]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv

# Load .env from the current working directory or notebook directory.
load_dotenv()
load_dotenv(Path.cwd() / ".env")


def _read_setting(primary: str, *aliases: str, secret: bool = False) -> str:
    """Read a required setting from env vars, falling back to a notebook prompt."""
    for key in (primary, *aliases):
        value = os.getenv(key)
        if value:
            os.environ[primary] = value
            return value

    prompt = f"Enter {primary}: "
    value = getpass.getpass(prompt) if secret else input(prompt)
    if not value:
        raise RuntimeError(f"Missing required setting: {primary}")
    os.environ[primary] = value
    return value


OPENAI_API_KEY = _read_setting("OPENAI_API_KEY", secret=True)
TAVILY_API_KEY = _read_setting("TAVILY_API_KEY", secret=True)
DB_USER = _read_setting("DB_USER", "ORACLE_USER")
DB_PASSWORD = _read_setting("DB_PASSWORD", "ORACLE_PASSWORD", secret=True)
DB_CONNECT_STRING = _read_setting("DB_CONNECT_STRING", "ORACLE_DSN")

# Jupyter runs an async event loop already; this lets us call async SDK methods cleanly.
import nest_asyncio
nest_asyncio.apply()

print("Configuration loaded from environment variables or prompts.")


Configuration loaded from environment variables or prompts.


## 3. Connect to Oracle AI Database and initialise the memory client

`OracleAgentMemory` is the governed memory client. In this notebook, automatic extraction is disabled so the separation is explicit:

- **Session items** are short-term OpenAI Agents SDK conversation state used for replay.
- **Research findings** are durable memories written only through the `save_research_finding` tool.

We use `text-embedding-3-small` for semantic search over durable findings.

> **💡 Key insight:** `schema_policy` controls how the library interacts with the DB on startup. `REQUIRE_EXISTING` never issues DDL. `CREATE_IF_NECESSARY` fills in missing objects and is safer for demos and development. `RECREATE` drops and rebuilds objects and should not be the default in a shared schema.

In [3]:
import oracledb
from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.dbschemapolicy import SchemaPolicy

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database.")

memory_client = OracleAgentMemory(
    connection=connection,
    embedder="text-embedding-3-small",   # OpenAI-compatible; 1536 dims
    extract_memories=False,               # durable findings are saved explicitly via tool calls
    schema_policy=SchemaPolicy.CREATE_IF_NECESSARY,
    table_name_prefix="genome_",         # isolates this example's tables from others
)
print("Memory client ready with safe schema policy:", SchemaPolicy.CREATE_IF_NECESSARY.value)

Connected to Oracle AI Database.
Memory client ready with safe schema policy: create_if_necessary


## 4. Register the research user and agent

Use a unique run scope so rerunning the notebook does not mix old demo memories with the current run. This keeps `CREATE_IF_NECESSARY` safe while avoiding stale long-term findings from previous executions.

In [4]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
USER_ID = f"genome-researcher-{RUN_ID}"
AGENT_ID = f"deep-research-agent-{RUN_ID}"

for create_fn, eid, info in [
    (memory_client.add_user, USER_ID, "Genomics researcher exploring human disease associations."),
    (memory_client.add_agent, AGENT_ID, "Deep-research agent with web search and long-term memory."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

print(f"Run scope: {RUN_ID}")

Registered: genome-researcher-20260608153949
Registered: deep-research-agent-20260608153949
Run scope: 20260608153949


## 5. Implement the `Session` protocol on top of Oracle AI Agent Memory

The OpenAI Agents SDK `Session` protocol handles **short-term session persistence**: the runner asks for previous items and appends new items during the agent loop.

In this implementation, session items are stored as Oracle Agent Memory `message` records with `metadata={"session_item": true}`. They are scoped by `session_id`, `user_id`, and `agent_id`, and they are used only for replaying the conversation. Durable long-term findings are stored separately as `memory` records through the `save_research_finding` tool.

> **Important:** Raw model/tool messages can be large and noisy. They should not appear as long-term research findings unless a tool deliberately promotes a concise conclusion.

In [5]:
import json
from typing import Optional
from agents.memory.session import SessionABC


class OracleAgentMemorySession(SessionABC):
    """Session backend that persists OpenAI Agents SDK items in Oracle AI Database.

    Session items are short-term replay state stored as message records. Durable
    research findings are saved separately via the save_research_finding tool.
    """

    def __init__(self, session_id: str, client: OracleAgentMemory, user_id: str, agent_id: str):
        self.session_id = session_id
        self._client = client
        self._store = client._store
        self._user_id = user_id
        self._agent_id = agent_id
        try:
            self._client.create_thread(
                thread_id=session_id,
                user_id=user_id,
                agent_id=agent_id,
                extract_memories=False,
            )
        except ValueError:
            pass  # thread already exists

    async def get_items(self, limit: Optional[int] = None) -> list:
        records = self._store.list(
            "message",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=limit if limit else 10_000,
            metadata_filter={"session_item": True},
        )
        items = [json.loads(r.content) for r in records]
        return items[-limit:] if limit else items

    async def add_items(self, items: list) -> None:
        if not items:
            return
        self._store.add(
            contents=[json.dumps(item) for item in items],
            record_type="message",
            thread_ids=self.session_id,
            user_ids=self._user_id,
            agent_ids=self._agent_id,
            roles=[self._message_role(item) for item in items],
            metadata=[{"session_item": True, "session_id": self.session_id} for _ in items],
        )

    async def pop_item(self) -> Optional[dict]:
        records = self._store.list(
            "message",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=1,
            metadata_filter={"session_item": True},
        )
        if not records:
            return None
        last = records[-1]
        self._store.delete("message", last.id)
        return json.loads(last.content)

    async def clear_session(self) -> None:
        records = self._store.list(
            "message",
            thread_id=self.session_id,
            user_id=self._user_id,
            agent_id=self._agent_id,
            limit=10_000,
            metadata_filter={"session_item": True},
        )
        for r in records:
            self._store.delete("message", r.id)

    @staticmethod
    def _message_role(item: dict) -> str:
        role = item.get("role") if isinstance(item, dict) else None
        if role in {"user", "assistant", "system", "tool"}:
            return role
        if isinstance(item, dict) and item.get("type") == "function_call_output":
            return "tool"
        return "assistant"


print("OracleAgentMemorySession implemented with message records for session replay.")

OracleAgentMemorySession implemented with message records for session replay.


## 6. Define the agent's tools

The agent needs three tools:

1. **`tavily_search`** — live web search for up-to-date genomics sources
2. **`save_research_finding`** — persist a durable research note the agent can recall in future sessions
3. **`recall_research_findings`** — search the long-term memory for prior findings on a topic

Tools 2 and 3 are how the agent manages its **long-term** memory. They're distinct from the `Session` machinery, which handles short-term conversation history automatically.

> **📌 The `@function_tool` decorator** auto-generates the tool's JSON schema from Python type hints and parses the docstring for the description the LLM sees. Write informative docstrings — the LLM reads them.

In [6]:
from agents import function_tool
from tavily import TavilyClient

_tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@function_tool
def tavily_search(query: str, max_results: int = 5) -> str:
    """Search the live web for recent, authoritative genomics sources.

    Use this to pull current information from NCBI, OMIM, peer-reviewed journals,
    and other scientific sources. Prefer this over relying on model-internal knowledge
    for facts about specific genes, mutations, or recent studies.

    Args:
        query: Natural-language search query.
        max_results: How many results to return (1-10).
    """
    resp = _tavily.search(
        query=query, max_results=max_results,
        search_depth="advanced", include_answer=True,
    )
    lines = [f"Answer: {resp.get('answer', '')}"]
    for r in resp.get("results", []):
        lines.append(f"- {r['title']} ({r['url']})\n  {r['content'][:300]}")
    return "\n".join(lines)


@function_tool
def save_research_finding(topic: str, finding: str) -> str:
    """Persist a durable research finding the agent should recall in future sessions.

    Use this for claims worth remembering across sessions, for example: "BRCA1
    pathogenic variants are associated with substantially elevated lifetime breast
    cancer risk." Do not use this for conversational acknowledgements, raw tool
    output, or speculation.

    Args:
        topic: Short topic tag (e.g. 'BRCA1', 'TP53 mutations').
        finding: The finding to remember, ideally one sentence.
    """
    memory_id = memory_client.add_memory(
        f"[{topic}] {finding}",
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata={"topic": topic, "kind": "research_finding", "durable": True},
    )
    return f"Saved durable research finding (id={memory_id})."


@function_tool
def recall_research_findings(query: str, max_results: int = 5) -> str:
    """Search prior durable research findings for information relevant to a query.

    Use this at the start of a research task to check what is already known.
    Results are ranked by semantic similarity.

    Args:
        query: Natural-language query describing what you want to recall.
        max_results: How many findings to return.
    """
    results = memory_client.search(
        query,
        user_id=USER_ID,
        agent_id=AGENT_ID,
        exact_user_match=True,
        exact_agent_match=True,
        max_results=max_results,
        record_types=["memory"],
    )
    if not results:
        return "(no prior durable findings)"
    return "\n".join(
        f"- {r.content}  [distance={r.distance:.3f}]" for r in results
    )


print("Tools defined: tavily_search, save_research_finding, recall_research_findings")

Tools defined: tavily_search, save_research_finding, recall_research_findings


## 7. Construct the research agent

The agent's `instructions` are its system prompt — the place to encode the behaviour you want. For a deep-research agent, the instructions should emphasise:

1. **Recall before research** — check long-term memory before hitting the web
2. **Cite sources** — so findings are traceable
3. **Save what's worth remembering** — commit durable conclusions explicitly

> **💡 Key insight:** A deep-research agent's output quality is largely determined by how well its instructions distinguish *research* (retrieve + synthesise) from *conversation* (respond + acknowledge). Encode that distinction in the system prompt.

In [7]:
from agents import Agent, Runner

INSTRUCTIONS = """You are a deep-research agent specialising in human genome exploration.

For every research question:
1. FIRST call `recall_research_findings` to check what is already known from prior sessions.
2. If the prior findings are insufficient or outdated, call `tavily_search` for current sources.
3. Synthesise a clear, cited answer. Prefer authoritative sources (NCBI, OMIM, PubMed).
4. Call `save_research_finding` for each durable conclusion worth remembering across sessions.
   Save one finding per call; keep findings atomic and one sentence long.
5. Present the final answer to the user with inline citations (URLs).

Do not save conversational acknowledgements as findings. Only save factual conclusions.
"""

research_agent = Agent(
    name="GenomeDeepResearcher",
    instructions=INSTRUCTIONS,
    tools=[tavily_search, save_research_finding, recall_research_findings],
    model="gpt-5.4",
)
print(f"Agent created: {research_agent.name}")

Agent created: GenomeDeepResearcher


## 8. Run a research session

We create a session (tied to a `thread_id` in the memory store) and run the agent over a sequence of research questions. Because we pass `session=session`, the SDK will automatically inject prior conversation items at the start of each turn and persist new items at the end.

> **📌 Separation of concerns.**
> - The `Session` persists the **raw conversation** (working memory).
> - The agent's `save_research_finding` tool persists **durable findings** (long-term memory).
> Both live in the same Oracle database but are accessed through different interfaces.

In [8]:
SESSION_ID = f"genome-session-{RUN_ID}-001"
session = OracleAgentMemorySession(
    session_id=SESSION_ID,
    client=memory_client,
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

research_questions = [
    "What is BRCA1 and what is its primary function in DNA repair?",
    "What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?",
    "How does BRCA1 interact with BRCA2 in homologous recombination?",
]

for i, q in enumerate(research_questions, 1):
    print(f"\n{'=' * 70}\nQ{i}: {q}\n{'=' * 70}")
    result = await Runner.run(research_agent, q, session=session)
    print(result.final_output)


Q1: What is BRCA1 and what is its primary function in DNA repair?
BRCA1 is a human tumor suppressor gene that encodes the BRCA1 protein, a key guardian of genome stability. Its primary role in DNA repair is helping repair DNA double-strand breaks through homologous recombination, a high-fidelity repair pathway. BRCA1 helps initiate DNA end resection and supports RAD51-mediated repair at damage sites. [NCBI GeneReviews](https://www.ncbi.nlm.nih.gov/books/NBK1247), [PMC review](https://pmc.ncbi.nlm.nih.gov/articles/PMC3599463)

In short: **BRCA1 mainly helps cells accurately repair broken DNA using homologous recombination.**

Q2: What is the typical lifetime breast cancer risk associated with pathogenic BRCA1 variants?
Pathogenic **BRCA1** variants are typically associated with a **~55%–72% lifetime breast cancer risk by age 70**, with **~65%** often quoted as a representative estimate. [NCBI GeneReviews](https://www.ncbi.nlm.nih.gov/books/NBK1247)

So, in short: **about two-thirds lif

## 9. Inspect what the agent remembered

At this point the agent has accumulated two different kinds of records:

- **Short-term session items**: raw OpenAI Agents SDK items that replay the conversation for the current session.
- **Durable research findings**: concise conclusions the agent deliberately saved through `save_research_finding`.

The two lists should look different. Session items may include user messages, tool calls, and tool outputs. Durable findings should be short, factual, reusable research notes — not raw `call_id` JSON or full tool output.

In [9]:
# Short-term: conversation items replayed each turn.
session_items = await session.get_items()
print(f"Session items: {len(session_items)}")
for it in session_items[:3]:
    print(f"  - {str(it)[:120]}...")

# Long-term: durable findings the agent deliberately chose to save.
findings = await memory_client.search_async(
    "BRCA1 function and risk",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    exact_user_match=True,
    exact_agent_match=True,
    max_results=10,
    record_types=["memory"],
)
print(f"\nDurable research findings: {len(findings)}")
for r in findings:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

raw_json_leaks = [r for r in findings if str(r.content).lstrip().startswith("{")]
print(f"\nRaw session/tool outputs in durable findings: {len(raw_json_leaks)}")

Session items: 28
  - {'content': 'What is BRCA1 and what is its primary function in DNA repair?', 'role': 'user'}...
  - {'arguments': '{"query":"BRCA1 gene identity and primary function in DNA repair", "max_results": 5}', 'call_id': 'call_C...
  - {'call_id': 'call_CYgO3zKYmIsqje4W02m5fI5z', 'output': '(no prior durable findings)', 'type': 'function_call_output'}...

Durable research findings: 5
  - [BRCA1] BRCA1 encodes a tumor suppressor protein that helps maintain genomic stability by coordinating repair of DNA double-strand breaks.  [distance=0.270]
  - [BRCA1 DNA repair] BRCA1’s primary DNA repair function is to promote homologous recombination repair of DNA double-strand breaks, in part by facilitating end resection and RAD51-mediated repair.  [distance=0.298]
  - [BRCA1 cancer risk] Women with a pathogenic germline BRCA1 variant have an estimated lifetime breast cancer risk of about 55% to 72% by age 70, with many summaries citing approximately 65%.  [distance=0.322]
  - [BRCA

## 10. Verify continuity — resume a fresh session with the same memory store

The real test of a memory-aware agent is whether it can pick up where a prior session left off. Let's create a *new* session (simulating a separate process or later day) and ask a question that builds on prior durable findings.

If the agent recalls BRCA1 findings from the previous run without depending on the previous session transcript, we've demonstrated end-to-end durable memory continuity.

In [10]:
# Simulate a fresh session (new session_id, but same run-scoped user/agent).
FRESH_SESSION_ID = f"genome-session-{RUN_ID}-002"
fresh_session = OracleAgentMemorySession(
    session_id=FRESH_SESSION_ID,
    client=memory_client,
    user_id=USER_ID,
    agent_id=AGENT_ID,
)

followup = "Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening."

print(f"FRESH SESSION — Q: {followup}\n")
result = await Runner.run(research_agent, followup, session=fresh_session)
print(result.final_output)

FRESH SESSION — Q: Based on what you already know about BRCA1, explain how a patient with a pathogenic BRCA1 variant might be counselled about screening.

A patient with a pathogenic **BRCA1** variant would usually be counselled that screening is **earlier, more intensive, and often combined with discussion of risk-reducing surgery**.

- **Breast cancer screening:** typically **annual breast MRI from age 25** and **annual mammography from age 30**, plus **clinical breast exam every 6–12 months**. Counseling should explain that MRI is especially important because BRCA1-associated breast cancer risk starts younger than average. (NCBI GeneReviews: https://www.ncbi.nlm.nih.gov/books/NBK1247 ; NCI PDQ: https://www.ncbi.nlm.nih.gov/books/NBK589498/)
- **Ovarian cancer risk:** patients should be told that **screening for ovarian cancer is limited in effectiveness**, so counseling often focuses on **risk-reducing bilateral salpingo-oophorectomy around age 35–40 or after childbearing** rather t

## 11. Cleanup (optional)

Cleanup is intentionally commented out. Uncomment it only when you want to remove the current run's session items and durable findings from your demo schema.

The setup above uses `CREATE_IF_NECESSARY`, so rerunning the notebook should not drop and recreate the schema by default. Each run uses a unique `RUN_ID` to avoid mixing current results with stale demo memories.

In [11]:
# await session.clear_session()
# await fresh_session.clear_session()
# for r in await memory_client.search_async("BRCA", user_id=USER_ID, agent_id=AGENT_ID, exact_user_match=True, exact_agent_match=True, max_results=100, record_types=["memory"]):
#     memory_client.delete_memory(r.record.id)
# print("Cleaned up current run.")

connection.close()
print("Connection closed.")

Connection closed.


## Key Takeaways

> **🎯 1. The `Session` protocol is the clean integration point.** Four async methods (`get_items`, `add_items`, `pop_item`, `clear_session`) is all the OpenAI Agents SDK needs to plug in a custom session backend. You don't have to modify the runner — you implement the protocol and pass an instance via `session=`.

> **🎯 2. Session persistence is not the same as long-term memory.** The session handles short-term conversation replay for the current agent loop. Durable findings should be concise conclusions written through explicit tools, not raw conversation or tool output dumped into long-term memory.

> **🎯 3. Instructions steer memory usage.** A deep-research agent must be explicitly told to *recall before researching* and *save durable conclusions*. Otherwise the LLM will treat memory as optional decoration and the store will fill up with low-value content.

> **🎯 4. Use safe schema defaults.** `CREATE_IF_NECESSARY` is safer for demos and development than a destructive recreate policy. Reserve destructive schema reset paths for isolated demo schemas.

> **🎯 5. Oracle AI Database gives you one governed substrate for both tiers.** Short-term session items and durable findings can live in one backend while remaining separate record types with different lifecycles.

> **🎯 6. Continuity is testable.** A new `session_id` with the same `user_id` / `agent_id` should produce an agent that reasons over prior durable findings. That's the simplest end-to-end test of a memory-aware agent.